In [6]:
import OrcFxAPI
from pathlib import Path
import numpy as np

owd_file = Path(r"C:\Users\verav\Desktop\Studie\Afstuderen\PHASE2_ORCA\OW_data\Harlequin_mesh2_fixed_V2_owd.owd")
owr_file = Path(r"C:\Users\verav\Desktop\Studie\Afstuderen\PHASE2_ORCA\OW_results\Mesh_DecayCalibration_fixed_fine_V2.owr")
xlsx_file = Path(r"C:\Users\verav\Desktop\Studie\Afstuderen\PHASE2_ORCA\XLSX_results\Mesh_DecayCalibration_fixed_fine_V2.xlsx")


In [7]:
water_depth = 30.0  # m
wave_periods_coarse = [1.1,1.2,1.3,1.4,1.5,1.6,1.7,1.8,1.9]

wave_periods_fine = np.arange(2, 5.01, 0.01)

wave_periods_rest = [5.1, 5.2, 5.3, 5.4 ,5.5, 5.6, 5.7, 5.8, 5.9, 6.0, 6.1, 6.2, 6.3, 6.4, 6.5, 6.6, 6.7, 6.8, 6.9, 7.0,
                7.1, 7.2, 7.3, 7.4, 7.5, 7.6, 7.7, 7.8, 7.9, 8.0, 8.5, 9, 9.5, 10, 10.5, 11, 11.5, 12, 13.5, 14, 14.5, 15, 15.5, 16, 17, 18, 19, 20 ]  # jouw rest

wave_periods = np.concatenate([
    wave_periods_coarse,
    wave_periods_fine,
    wave_periods_rest
])
wave_headings = [0.0, 45.0, 90.0, 135.0, 180.0]
   # deg

mass = 60.75  # ton, alleen goed als jouw model ook ton verwacht
com_x = 0.0
com_y = 0.0
com_z = 10.06
# voorbeeld inertia matrix rond CoM
Ixx = 4659.53 
Iyy = 4659.53
Izz = 7596.09
Ixy = 0.0
Ixz = 0.0
Iyz = 0.0


# =========================
# HELPER FUNCTIONS
# =========================
def set_value(obj, name, value):
    """
    Zet een OrcaWave data item.
    De exacte data-itemnamen moet je uit OrcaWave halen met F7.
    """
    try:
        obj[name] = value
        print(f"Set: {name} = {value}")
    except Exception as e:
        print(f"Kon data item niet zetten: {name}")
        print(f"  Waarde: {value}")
        print(f"  Fout: {e}")


def print_validation(diff):
    info = getattr(diff, "ValidationInformationText", "")
    warnings = getattr(diff, "ValidationWarningText", "")
    errors = getattr(diff, "ValidationErrorText", "")

    print("\n=== VALIDATION INFO ===")
    print(info if info else "(geen info)")

    print("\n=== VALIDATION WARNINGS ===")
    print(warnings if warnings else "(geen warnings)")

    print("\n=== VALIDATION ERRORS ===")
    print(errors if errors else "(geen errors)")

    return errors


# =========================
# MAIN
# =========================
diff = OrcFxAPI.Diffraction()
diff.LoadData(str(owd_file))

# -------------------------------------------------
# BELANGRIJK:
# De namen hieronder zijn PLACEHOLDERS / typische structuur.
# Vervang ze met de exacte OrcaWave data names via F7 in de GUI.
# -------------------------------------------------

# Environment
diff.SetData("WaterDepth", 0, 30.0)

# Wave periods
# Vaak moet je eerst een count zetten en daarna de tabel vullen
diff.SetData("NumberOfPeriodsOrFrequencies", 0, len(wave_periods))
for i, T in enumerate(wave_periods):
    diff.SetData(f"PeriodOrFrequency", i, T)

# Wave headings
diff.SetData("NumberOfWaveHeadings", 0, len(wave_headings))
for i, hdg in enumerate(wave_headings):
    diff.SetData(f"WaveHeading", i, hdg)


diff.SetData("BodyInertiaSpecifiedBy", 0, "Matrix (for a general body)")
diff.SetData("BodyInertiaTensorOriginType", 0, "Centre of mass")
# Body inertia / mass properties
# Let op: body index / naam kan anders zijn in jouw model
diff.SetData("BodyCentreOfMassX", 0, com_x)
diff.SetData("BodyCentreOfMassY", 0, com_y)
diff.SetData("BodyCentreOfMassZ", 0, com_z)

diff.SetData("BodyMass", 0, mass)



diff.SetData("BodyInertiaTensorRx", 0, Ixx)   # xx
diff.SetData("BodyInertiaTensorRy", 1, Iyy)   # yy
diff.SetData("BodyInertiaTensorRz", 2, Izz)   # zz


print("Rx:", diff.BodyInertiaTensorRx[0])
print("Ry:", diff.BodyInertiaTensorRy[1])
print("Rz:", diff.BodyInertiaTensorRz[2])


# Optioneel: output settings
# Ook hier weer: exacte namen via F7
# set_value(diff, "CalculateDisplacementRAOs", "Yes")
# set_value(diff, "CalculateLoadRAOs", "Yes")
# set_value(diff, "CalculateAddedMassAndDamping", "Yes")
# set_value(diff, "CalculateWaveDriftQTFs", "No")

# Validation
errors = print_validation(diff)
if errors:
    raise RuntimeError("Model heeft validation errors. Fix eerst de data-itemnamen of invoer.")



Rx: 4659.53
Ry: 4659.53
Rz: 7596.09

=== VALIDATION INFO ===
('Estimated peak memory required during calculation: 94,9 MiB per thread.',)

=== VALIDATION WARNINGS ===
('Calculation mesh: the following panels are constructed from non-planar data in the mesh file: 4-8 19-23 32-38 47-53 63-68 77-83 92 94-98 109-113 124-128 137 139-143 152-158 168-173 182-188 197-203 214-218 229-233 244-248 259-263 272-278 287-293 303-308 317-323 332 334-338 349-353 364-368 377 379-383 392-398 408-413 422-428 437-443 454-458 469-473 482-488 497-501 503 513 515-518 527-530 532 533 542-548 557-563 574-578 587-589 591-593 602-608 618 620-623 633-638 648-653 662-668 678-683 693-697 707-709 711-713 722-728 737-741 743 752 753 755-758 767-770 772 773 782-788 797-803 813-818 827-829 831-833 842-848 858 860-863 873-878 888-893 902-908 918-923 933-937 947-949 951-953 962-964 966-968 978-982 993-998 1007-1013 1023-1028 1038-1043 1053 1055-1058 1067-1073 1082-1084 1086-1088 1098-1103 1112-1118 1127-1133 1142-1145 114

In [8]:
# Run calculation
print("\n=== START CALCULATION ===")
diff.Calculate()
print("=== CALCULATION DONE ===")

# Save outputs
diff.SaveResults(str(owr_file))
diff.SaveResultsSpreadsheet(str(xlsx_file))

print("\nBestanden opgeslagen:")
print(f"  Results: {owr_file}")
print(f"  Spreadsheet: {xlsx_file}")


=== START CALCULATION ===


=== CALCULATION DONE ===

Bestanden opgeslagen:
  Results: C:\Users\verav\Desktop\Studie\Afstuderen\PHASE2_ORCA\OW_results\Mesh_DecayCalibration_fixed_fine_V2.owr
  Spreadsheet: C:\Users\verav\Desktop\Studie\Afstuderen\PHASE2_ORCA\XLSX_results\Mesh_DecayCalibration_fixed_fine_V2.xlsx
